# NB0 · Prérequis — vérification de l'environnement

**Objectif :** valider que votre environnement est prêt pour les Jours 1 et 2 de la formation.

> Ce notebook doit s'exécuter **entièrement sans erreur** avant le Jour 1.
>
> Durée estimée : ~5 minutes (installation comprise).


---

## 1 · Installation des dépendances (via UV)

On utilise **[uv](https://github.com/astral-sh/uv)** pour créer un environnement propre et reproductible.
Si `uv` n'est pas encore installé :

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Ensuite, depuis la racine du dépôt :

```bash
uv venv .venv
uv pip install -r requirements-j1j2.txt
```

Ou directement depuis ce notebook (cellule suivante) :

In [ ]:
# Installe toutes les dépendances J1 + J2 dans le venv courant
# Prérequis : uv installé (https://github.com/astral-sh/uv)
import subprocess, sys

pkgs = [
    # Core
    "python-dotenv",
    "requests",
    "mistralai>=1.0",
    "mistralai-workflows",
    "pydantic>=2",
    # J1 · T1 Workflows
    "langgraph>=0.2",
    "langchain-mistralai",
    "langchain-core",
    # J1 · T2 Multi-agents
    "instructor",
    "litellm",
    "ragas",
    "pandas",
    "matplotlib",
    "gpxpy",
    # J2 · T3 MCP
    "mcp",
    "fastmcp",
    "python-frontmatter",
    # J2 · T4 GraphRAG
    "llama-index-core",
    "llama-index-llms-mistralai",
    "beautifulsoup4",
    "rank-bm25",
    "numpy",
    "networkx",
    "lightrag-hku",
]

result = subprocess.run(
    ["uv", "pip", "install"] + pkgs,
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("ERREUR:", result.stderr[-1000:])
else:
    print("Installation OK")

---

## 2 · Vérification des versions

In [ ]:
import sys, importlib.metadata
print(f"Python : {sys.version.split()[0]}")

checks = [
    "mistralai",
    "langgraph",
    "langchain-core",
    "llama-index-core",
    "ragas",
    "fastmcp",
    "pydantic",
]

ok, fail = [], []
for pkg in checks:
    try:
        ver = importlib.metadata.version(pkg)
        ok.append(f"  OK  {pkg:<25} {ver}")
    except importlib.metadata.PackageNotFoundError:
        fail.append(f"  KO  {pkg:<25} non installé")

for line in ok:
    print(line)
if fail:
    print()
    for line in fail:
        print(line)
    print("\nRelancez la cellule d'installation.")
else:
    print("\nToutes les librairies sont disponibles.")

---

## 3 · Clé API Mistral

Créez un fichier `.env` à la racine du projet :

```
MISTRAL_API_KEY=sk-••••••••••••••••
```

Ne commitez jamais ce fichier (ajoutez `.env` à votre `.gitignore`).

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

key = os.environ.get("MISTRAL_API_KEY", "")
if key:
    print(f"MISTRAL_API_KEY : OK ({key[:8]}…)")
else:
    print("MISTRAL_API_KEY absente — ajoutez-la dans .env avant les ateliers.")

---

## 4 · Test rapide de l'API

In [ ]:
import os
from dotenv import load_dotenv
from mistralai.client import Mistral

load_dotenv()
key = os.environ.get("MISTRAL_API_KEY", "")
server_url = os.environ.get("MISTRAL_SERVER_URL")  # serveur dédié si configuré

if not key:
    print("Clé API absente, test ignoré.")
else:
    kwargs = {"api_key": key}
    if server_url:
        kwargs["server_url"] = server_url
        print(f"Serveur : {server_url}")
    client = Mistral(**kwargs)
    try:
        resp = client.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": "Réponds juste : OK"}],
        )
        print("Réponse API :", resp.choices[0].message.content)
        print("Tokens consommés :", resp.usage.total_tokens)
    except Exception as e:
        print(f"Appel API échoué : {e}")

---

> **Tout vert ?** Votre environnement est prêt. Passez au Jour 1.
>
> En cas de problème, vérifiez que `uv` est installé et que vous exécutez le notebook dans le bon venv (kernel Python du `.venv`).